In [38]:
%pip install -q --upgrade --force-reinstall langchain-groq langchain-core
!pip install grandalf

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-genai 2.12.1 requires tenacity<9.2.0,>=8.2.3, but you have tenacity 9.2.1 which is incompatible.
google-genai 2.12.1 requires websockets<17.0,>=13.0.0, but you have websockets 17.4 which is incompatible.
google-adk 2.7.1 requires websockets<16,>=15.0.1, but you have websockets 17.4 which is incompatible.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.29.0 which is incompatible.


In [61]:
from langchain_groq import ChatGroq
from pathlib import Path
import os
from google.colab import userdata
from langchain_core.prompts  import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda , chain as chain_decorator 
from operator import itemgetter

In [40]:
def _read_env_file(name, filename="secrets.env"):
    """Look for KEY=value in secrets.env in the working dir or /content."""
    for folder in (Path.cwd(), Path("/content")):
        path = folder / filename
        if path.is_file():
            for line in path.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, _, val = line.partition("=")
                if key.strip().removeprefix("export ").strip() == name:
                    return val.strip().strip('"').strip("'")
    return None

def get_secret(name, required=True):
    value = None
    try:
        value = userdata.get(name)      # works in the Colab web UI
    except Exception:
        pass                            # timeout / not found / no access
    if not value:
        value = _read_env_file(name)    # fallback: uploaded secrets.env
    if value:
        os.environ[name] = value
    elif required:
        raise RuntimeError(
            f"{name} not found in Colab Secrets or secrets.env. "
            "Upload secrets.env to the Colab server and re-run."
        )
    return value

groq_key = get_secret("GROQ_API_KEY")


print("GROQ key loaded:", bool(groq_key))


GROQ key loaded: True


In [41]:
llm = ChatGroq(
        model = "openai/gpt-oss-120b",
        model_kwargs= {"seed" : 350 },
        temperature= 0,
        max_tokens= 250
)

Runnable pipeline :

In [42]:
template = ChatPromptTemplate.from_messages([
    ('system' , 'you are a lively chatbot that answers in a lively way'),
    ('human' , "it's cold weather tonight isn't it ?")
])

In [43]:
chaining = template | llm | StrOutputParser()

response = chaining.invoke({})

print(response)

Brrr! 🌬️ Tonight’s chill is definitely giving the night sky a frosty vibe—perfect for cozy blankets, hot cocoa, and maybe a good book or a binge‑watch session! How are you staying warm? ☕🧣✨


Batching

In [44]:
batch_template = ChatPromptTemplate.from_messages([
    ('system' , 'you will be given two words , use them in one sentence'),
    ('human' , "{word_1} , {word_2}")
])

In [45]:
chain = batch_template | llm | StrOutputParser()

In [46]:
%time
print(chain.invoke({'word_1' : 'hello' , 'word_2' : 'bright'}))
%time
print(chain.invoke({'word_1' : 'shuffle' , 'word_2' : 'proud'}))

CPU times: user 2 µs, sys: 0 ns, total: 2 µs
Wall time: 4.29 µs
Hello, bright sunrise, you’ve already painted the sky with hope.
CPU times: user 3 µs, sys: 0 ns, total: 3 µs
Wall time: 4.77 µs
I felt proud as I watched the dancers shuffle across the stage, each step perfectly timed to the rhythm.


In [47]:
%time
chain.batch(inputs= [
    {'word_1' : 'laptop' , 'word_2' : 'cat'},
    {'word_1' : 'sand' , 'word_2' : 'phone'}
])

CPU times: user 2 µs, sys: 0 ns, total: 2 µs
Wall time: 4.29 µs


['The cat curled up on the warm laptop, purring as the screen glowed softly.',
 'The phone slipped from his hand and landed in the sand, sending a tiny cloud of grains into the air.']

Stream:

In [48]:
stream_template = ChatPromptTemplate.from_messages([
    ('system' , 'you will be given a topic, generate some point on it'),
    ('human' , '{topic}')
])

In [49]:
chaining = stream_template | llm | StrOutputParser()
response = chaining.stream({'gta 6'})

#next(response)

for i in response:
   print(i, end= '')



**Grand Theft Auto VI – Key Talking Points**

| Topic | What We Know / What’s Expected |
|-------|--------------------------------|
| **Release Window** | • Rockstar has hinted that the next main‑line entry will launch **sometime in 2026‑2027**. <br>• Development milestones (engine upgrades, QA) suggest a Q4 2026 release is plausible, but no official date has been confirmed. |
| **Setting** | • Rumors point to a **dual‑city map** (Vice City + a new “Caribbean” metropolis) or a **reimagined Miami‑style region** with expanded rural outskirts. <br>• Expect a **modern, hyper‑realistic world** with dynamic weather, day‑night cycles, and a living economy. |
| **Story & Protagonists** | • Early

RunnablePassthrough

In [50]:
RunnablePassthrough().invoke({'topic' : 'Modern AI '})

{'topic': 'Modern AI '}

In [51]:
template1 = ChatPromptTemplate.from_messages([
    ('system', '''
    generate a list of synonyms for given word like this:
    [a,b,c,d]
    '''),
    ('human', '{word}')
])

template2 = ChatPromptTemplate.from_messages([
    ('system', "you'll be given a list , pick a word that suits you"),
    ('human', '{list}')
])

In [52]:
chain_long = template1 | llm | StrOutputParser() | {'list' : RunnablePassthrough()}  | template2 | llm | StrOutputParser()

chain_long.invoke({'side'})

'**Perspective** – it captures how I view information from many angles and help you see things from different viewpoints.'

In [53]:
chain_long.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
            *              
            *              
            *              
      +----------+         
      | ChatGroq |         
      +----------+         
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
     +-------------+       
     | Passthrough |       
     +-------------+       
            *              
            *              
            *              
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
            *              
            *              
            *       

Runnable Parallel 

In [54]:
template_parallel_1 = ChatPromptTemplate.from_messages([
    ('system', 'create a question on a given topic'),
    ('human', '{topic}')
])

template_parallel_2 = ChatPromptTemplate.from_messages([
    ('system', 'generate a sarcastic sentence on given word'),
    ('human', '{word}')
])



In [55]:
chain1 = template_parallel_1 | llm | StrOutputParser()
chain2 = template_parallel_2 | llm | StrOutputParser()

chain_parallel = RunnableParallel({'parallel_1':  chain1, 'parallel_2' : chain2})

chain_parallel.invoke({'topic':'headphones', 'word' : 'car'})

{'parallel_1': '**Question:**  \n\n*How do the design differences between open‑back and closed‑back headphones affect sound quality, listener isolation, and typical use‑case scenarios?*  \n\nProvide a detailed comparison that covers:\n\n1. **Acoustic characteristics** (frequency response, soundstage, leakage).  \n2. **Physical isolation** (how well each type blocks external noise and prevents sound from leaking out).  \n3. **Practical applications** (ideal environments and activities for each style, such as studio mixing, gaming, commuting, or casual listening).  \n\nUse specific examples of popular headphone models to illustrate your points.',
 'parallel_2': 'Oh, because nothing says “I’ve got my life together” quite like a car that spends more time in the shop than on the road.'}

Runnable lambda:

In [56]:
sum = RunnableLambda(lambda x : (x +2))

chain_lambda1 = sum | RunnableLambda(lambda x : (x * 2))

chain_lambda1.invoke(2)

8

In [57]:
a = []

def generate_array(x):
    for element in x:
        a.append(element + 2)
    return a

def sum_array(y):
    b = 0
    for element in y:
        b = b + element
    return b

chain_lambda2 = RunnableLambda(generate_array) | RunnableLambda(sum_array)

chain_lambda2.invoke([2,3])

9

In [60]:
@chain_decorator
def increase_num(x):
    return  x + 2

@chain_decorator
def power_num(y):
    return y **2

chain_lambda3 = increase_num | power_num

chain_lambda3.invoke(3)

25

ItemGetter

In [64]:
dictionary = {'book' : 'abcd' , 'projects' : 'def'}

itemgetter('book')(dictionary)

'abcd'